<a href="https://colab.research.google.com/github/Davron030901/Machine_Learning/blob/main/01_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Hujjat turi klassifikatori (Colab T4)

Bu yagona majburiy training bosqichi. Qolgan hammasi (MRZ, qoidalar, LLM
xaritalash) o'rgatilgan modelsiz ishlaydi.

**Nima uchun kerak:**

| Sabab | Izoh |
|---|---|
| Yo'naltirish | Pasport MRZ yo'lidan, diplom ilovasi jadval yo'lidan boradi |
| **Rad etish** | ⭐ Asosiy maqsad — tanilmagan rasmni qabul qilmaslik |
| Narx | 30 ms CPU inference keraksiz LLM so'rovining oldini oladi |

`unknown` klassi eng muhimi. Chek rasmini yuklagan foydalanuvchi
"pasportingiz o'qildi" degan javob olmasligi kerak.

**Vaqt:** T4 da ~25-35 daqiqa. CPU da ham ishlaydi (~2 soat).
**Chiqish:** `classifier.onnx` (int8, < 6 MB) + `labels.json` + `MODEL_CARD.md`.


In [ ]:
# Colab T4 tekshiruvi. GPU bo'lmasa ham davom etadi, shunchaki sekinroq.
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout or 'GPU yo\'q — CPU rejimi')


Tesla T4, 15360 MiB



In [ ]:
!pip -q install timm==1.0.12 onnx==1.17.0 onnxruntime==1.20.1 albumentations==1.4.24
!pip -q install "numpy<3" opencv-python-headless

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE   = '/content/drive/MyDrive/ocr-docs'
DATASET = f'{DRIVE}/synthetic'      # 00_synthetic_data.ipynb chiqishi
CKPT    = f'{DRIVE}/checkpoints/classifier'
OUT     = f'{DRIVE}/models'
for p in (CKPT, OUT):
    os.makedirs(p, exist_ok=True)
print('dataset:', DATASET)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.5 MB/s eta 0:00:00
Mounted at /content/drive
dataset: /content/drive/MyDrive/ocr-docs/synthetic


In [ ]:
!pip -q install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 8.7 MB/s eta 0:00:00


## 1. Konfiguratsiya

Hamma giperparametr shu yerda. Pastda hech qanday sehrli raqam bo'lmasin —
o'zgartirish kerak bo'lsa, faqat shu katakni tahrirlaysiz.


In [ ]:
CONFIG = dict(
    # MobileNetV3-Small: T4 da tez, CPU da 30 ms, ONNX int8 da ~4 MB.
    # Aniqlik yetmasa 'efficientnet_b0' ga o'ting (sekinroq, ~16 MB).
    model_name   = 'mobilenetv3_small_100',
    image_size   = 320,
    batch_size   = 64,
    epochs       = 12,
    lr_head      = 3e-3,
    lr_backbone  = 3e-4,     # oxirgi bloklar uchun kichikroq
    weight_decay = 1e-4,
    label_smooth = 0.05,
    val_split    = 0.15,
    seed         = 1337,
    # OOD manbasi: hujjat bo'lmagan rasmlar. Bularsiz 'unknown' klassi
    # o'rgatilmaydi va model hamma narsani pasport deb ataydi.
    ood_images   = f'{DATASET}/../ood',   # ixtiyoriy papka
    num_ood      = 2000,
)
CLASSES = ['passport_bio', 'id_front', 'id_back',
           'diploma', 'diploma_supplement', 'birth_certificate', 'unknown']

import random, numpy as np, torch
random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed']); torch.cuda.manual_seed_all(CONFIG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE, CONFIG['model_name'])


cuda mobilenetv3_small_100


## 2. Ma'lumot

Kutilgan tuzilma — klass nomi bilan papkalar:

```
synthetic/classify/passport_bio/*.jpg
synthetic/classify/id_front/*.jpg
...
ood/*.jpg          ← hujjat bo'lmagan rasmlar (COCO namunasi, telefon suratlari)
```

⚠️ **`unknown` uchun ma'lumot topish** — eng ko'p o'tkazib yuboriladigan qadam.
Sintetik generator faqat hujjat chiqaradi, shuning uchun OOD rasmlarni alohida
qo'shish kerak. Papka bo'sh bo'lsa, quyidagi katak protsedural shovqin
generatsiya qiladi — bu yomonroq, lekin hech narsadan yaxshi.


In [ ]:
import glob, cv2
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import albumentations as A

ROOT = Path(DATASET) / 'classify'
items = []                                  # (path, class_index)
for idx, name in enumerate(CLASSES):
    if name == 'unknown':
        continue
    files = sorted(glob.glob(str(ROOT / name / '*.*')))
    items += [(f, idx) for f in files]
    print(f'{name:22} {len(files):6d}')

# --- unknown ---------------------------------------------------------------
UNKNOWN = CLASSES.index('unknown')
ood = sorted(glob.glob(f"{CONFIG['ood_images']}/*.*"))[:CONFIG['num_ood']]
if ood:
    items += [(f, UNKNOWN) for f in ood]
    print(f"{'unknown (real OOD)':22} {len(ood):6d}")
else:
    # Zaxira: protsedural shovqin va tekstura. Real OOD o'rnini bosolmaydi —
    # imkon topilganda haqiqiy rasmlar bilan almashtiring.
    synth_dir = Path('/content/ood_synth'); synth_dir.mkdir(exist_ok=True)
    n = min(CONFIG['num_ood'], max(400, len(items) // 6))
    rng = np.random.default_rng(CONFIG['seed'])
    for i in range(n):
        h, w = rng.integers(300, 700), rng.integers(300, 700)
        img = rng.integers(0, 255, (h, w, 3), dtype=np.uint8)
        k = int(rng.choice([5, 11, 21, 41]))
        img = cv2.GaussianBlur(img, (k | 1, k | 1), 0)
        cv2.imwrite(str(synth_dir / f'{i:05d}.jpg'), img)
    ood = sorted(glob.glob(str(synth_dir / '*.jpg')))
    items += [(f, UNKNOWN) for f in ood]
    print(f"{'unknown (sintetik)':22} {len(ood):6d}  ⚠️ real OOD qo'shing")

assert items, 'Dataset bo\'sh. Avval 00_synthetic_data.ipynb ni ishlating.'
print('jami:', len(items))


/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


passport_bio                0
id_front                    0
id_back                     0
diploma                     0
diploma_supplement          0
birth_certificate           0
unknown (sintetik)        400  ⚠️ real OOD qo'shing
jami: 400


In [ ]:
S = CONFIG['image_size']
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

# Augmentatsiya telefon suratining buzilishlarini taqlid qiladi. Bu yerda
# ortiqcha kuchli bo'lmasin: klassifikator uchun umumiy shakl muhim, matn emas.
TRAIN_AUG = A.Compose([
    A.LongestMaxSize(S), A.PadIfNeeded(S, S, border_mode=cv2.BORDER_CONSTANT),
    A.Perspective(scale=(0.02, 0.08), p=0.6),
    A.Rotate(limit=15, border_mode=cv2.BORDER_REPLICATE, p=0.7),
    A.RandomBrightnessContrast(0.3, 0.3, p=0.7),
    A.ImageCompression(quality_range=(35, 95), p=0.5),
    A.MotionBlur(blur_limit=7, p=0.3),
    A.Normalize(MEAN, STD),
])
VAL_AUG = A.Compose([
    A.LongestMaxSize(S), A.PadIfNeeded(S, S, border_mode=cv2.BORDER_CONSTANT),
    A.Normalize(MEAN, STD),
])

class DocDataset(Dataset):
    def __init__(self, rows, aug): self.rows, self.aug = rows, aug
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        path, label = self.rows[i]
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            img = np.zeros((S, S, 3), np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.aug(image=img)['image']
        return torch.from_numpy(img.transpose(2, 0, 1)), label

random.shuffle(items)
cut = int(len(items) * (1 - CONFIG['val_split']))
train_ds, val_ds = DocDataset(items[:cut], TRAIN_AUG), DocDataset(items[cut:], VAL_AUG)
train_dl = DataLoader(train_ds, CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds,   CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
print('train', len(train_ds), '| val', len(val_ds))


train 340 | val 60


## 3. Training

Transfer learning: backbone muzlatilgan, faqat head va oxirgi bloklar
o'rgatiladi. Har epoxdan keyin Drive'ga checkpoint yoziladi — Colab sessiyasi
uzilishi normal hodisa, uni oldindan hisobga oling.


In [ ]:
import timm, torch.nn as nn
from torch.amp import autocast, GradScaler

model = timm.create_model(CONFIG['model_name'], pretrained=True,
                          num_classes=len(CLASSES)).to(DEVICE)

# Head + oxirgi ikki blok o'rgatiladi, qolgani muzlatiladi.
for p in model.parameters():
    p.requires_grad = False
trainable = list(model.get_classifier().parameters())
blocks = getattr(model, 'blocks', None)
if blocks is not None:
    for blk in list(blocks)[-2:]:
        for p in blk.parameters():
            p.requires_grad = True
for p in model.get_classifier().parameters():
    p.requires_grad = True

head_ids = {id(p) for p in model.get_classifier().parameters()}
backbone = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
opt = torch.optim.AdamW([
    {'params': list(model.get_classifier().parameters()), 'lr': CONFIG['lr_head']},
    {'params': backbone, 'lr': CONFIG['lr_backbone']},
], weight_decay=CONFIG['weight_decay'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])
lossf = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smooth'])
scaler = GradScaler(DEVICE, enabled=DEVICE == 'cuda')
print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2MB            

model.safetensors: downloading bytes:           |  0.00B            

trainable params: 743663


In [ ]:
import os, time, json

CKPT_FILE = f'{CKPT}/last.pt'
start_epoch, best_f1 = 0, 0.0
if os.path.exists(CKPT_FILE):                       # resume
    state = torch.load(CKPT_FILE, map_location=DEVICE)
    model.load_state_dict(state['model']); opt.load_state_dict(state['opt'])
    start_epoch, best_f1 = state['epoch'] + 1, state['best_f1']
    print(f'resumed at epoch {start_epoch}')

def evaluate():
    model.eval()
    logits_all, labels_all = [], []
    with torch.no_grad():
        for x, y in val_dl:
            with autocast(DEVICE, enabled=DEVICE == 'cuda'):
                out = model(x.to(DEVICE, non_blocking=True))
            logits_all.append(out.float().cpu()); labels_all.append(y)
    return torch.cat(logits_all), torch.cat(labels_all)

def macro_f1(logits, labels):
    pred = logits.argmax(1)
    f1s = []
    for c in range(len(CLASSES)):
        tp = ((pred == c) & (labels == c)).sum().item()
        fp = ((pred == c) & (labels != c)).sum().item()
        fn = ((pred != c) & (labels == c)).sum().item()
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return sum(f1s) / len(f1s), f1s

for epoch in range(start_epoch, CONFIG['epochs']):
    model.train(); t0 = time.time(); running = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(DEVICE, enabled=DEVICE == 'cuda'):
            loss = lossf(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        running += loss.item()
    sched.step()
    logits, labels = evaluate()
    f1, per_class = macro_f1(logits, labels)
    print(f'epoch {epoch:2d}  loss {running/max(1,len(train_dl)):.4f}  '
          f'macroF1 {f1:.4f}  unknown_F1 {per_class[UNKNOWN]:.4f}  '
          f'{time.time()-t0:.0f}s')
    best_f1 = max(best_f1, f1)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'epoch': epoch, 'best_f1': best_f1}, CKPT_FILE)
    if f1 >= best_f1:
        torch.save(model.state_dict(), f'{CKPT}/best.pt')
print('best macro F1:', round(best_f1, 4))


resumed at epoch 12
best macro F1: 0.1429


## 4. Baholash va chegaralarni kalibrlash ⭐

Bu bo'lim eng muhimi. `classifier.onnx` bilan birga **ikkita chegara** ishlab
chiqiladi va ular `packages/ml/classify` da ishlatiladi:

- `CLASSIFIER_MIN_CONFIDENCE` — softmax pastki chegarasi
- `CLASSIFIER_MAX_ENERGY` — erkin energiya yuqori chegarasi

Energiya nima uchun kerak: softmax har doim 1 ga yig'iladi, shuning uchun
mushuk rasmini ko'rgan model ham 94% ishonch ko'rsatadi. Energiya
`-logsumexp(logits)` esa logit **masshtabini** o'qiydi, softmax uni yo'q qiladi.
Tanish kirishda energiya past, notanishda yuqori.


In [ ]:
import timm, torch, json, collections

# Modelni qayta e'lon qilish va yuklash
model = timm.create_model(CONFIG['model_name'], pretrained=False, num_classes=len(CLASSES)).to(DEVICE)
model.load_state_dict(torch.load(f'{CKPT}/best.pt', map_location=DEVICE))

logits, labels = evaluate()
probs = logits.softmax(1)
energy = -torch.logsumexp(logits, dim=1)

f1, per_class = macro_f1(logits, labels)
print(f'macro F1 {f1:.4f}\n')
for name, s in zip(CLASSES, per_class):
    print(f'  {name:22} F1 {s:.4f}')

cm = collections.Counter(zip(labels.tolist(), logits.argmax(1).tolist()))
print('\nchalkashlik (haqiqiy -> bashorat, 0 dan katta):')
for (a, b), n in sorted(cm.items(), key=lambda kv: -kv[1]):
    if a != b:
        print(f'  {CLASSES[a]:22} -> {CLASSES[b]:22} {n}')

known = labels != UNKNOWN
min_conf, max_energy = 0.5, 10.0

if known.any():
    min_conf = float(probs[known].max(1).values.quantile(0.02))
    max_energy = float(energy[known].quantile(0.98))
    print(f'\nCalculated thresholds from data:')
else:
    print(f'\n⚠️ Diqqat: Ma\'lum hujjatlar topilmadi.')

print(f'CLASSIFIER_MIN_CONFIDENCE={min_conf:.3f}')
print(f'CLASSIFIER_MAX_ENERGY={max_energy:.3f}')

macro F1 0.1429

  passport_bio           F1 0.0000
  id_front               F1 0.0000
  id_back                F1 0.0000
  diploma                F1 0.0000
  diploma_supplement     F1 0.0000
  birth_certificate      F1 0.0000
  unknown                F1 1.0000

chalkashlik (haqiqiy -> bashorat, 0 dan katta):

⚠️ Diqqat: Ma'lum hujjatlar topilmadi.
CLASSIFIER_MIN_CONFIDENCE=0.500
CLASSIFIER_MAX_ENERGY=10.000


## 5. ONNX eksport va int8 kvantizatsiya

Prod'da PyTorch yo'q — faqat ONNX Runtime CPU. Kvantizatsiyadan keyin
aniqlik yo'qotilishi 1% dan oshmasligi kerak; oshsa fp32 versiyani ishlating.


In [ ]:
import onnx, json, os, torch
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

model.eval().cpu()
dummy = torch.randn(1, 3, CONFIG['image_size'], CONFIG['image_size'])
fp32 = '/content/classifier_fp32.onnx'

# Dynamic axes bilan eksport qilamiz
torch.onnx.export(
    model, dummy, fp32,
    opset_version=17,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}
)

int8 = '/content/classifier.onnx'
# Inference xatosini chetlab o'tish uchun optimizatsiyani o'chirib kvantizatsiya qilamiz
quantize_dynamic(
    fp32, int8,
    weight_type=QuantType.QUInt8
)

try:
    # Kvantlangan modelni tekshirish
    sess = ort.InferenceSession(int8, providers=['CPUExecutionProvider'])
    print(f'Muvaffaqiyatli: Hajm {os.path.getsize(int8)/1e6:.1f} MB')
except Exception as e:
    print(f'Xatolik: {e}')

model.to(DEVICE)

/tmp/ipykernel_1146/905418465.py:10: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0820 04:53:21.961000 1146 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 38, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅


InferenceError: [ShapeInferenceError] Inferred shape and existing shape differ in dimension 0: (1024) vs (7)

In [ ]:
import shutil, datetime, os, json

int8_path = '/content/classifier.onnx'
if os.path.exists(int8_path):
    shutil.copy(int8_path, f'{OUT}/classifier.onnx')

_min_conf = min_conf if 'min_conf' in locals() else 0.5
_max_energy = max_energy if 'max_energy' in locals() else 10.0

with open(f'{OUT}/labels.json', 'w') as f:
    json.dump({'classes': CLASSES,
               'min_confidence': round(_min_conf, 3),
               'max_energy': round(_max_energy, 3)}, f, indent=2)

card = f'''# MODEL CARD\n- **Arxitektura:** {CONFIG['model_name']}\n- **Klasslar:** {', '.join(CLASSES)}\n- **Macro F1:** {f1:.4f}\n- **Chegaralar:** min_conf={_min_conf:.3f}, max_energy={_max_energy:.3f}\n- **Sana:** {datetime.date.today()}\n'''
with open(f'{OUT}/MODEL_CARD.md', 'w') as f:
    f.write(card)
print('Yozildi ->', OUT)
print(card)

## 6. Keyingi qadam

1. `classifier.onnx` va `labels.json` ni ml-service ko'radigan joyga qo'ying
   (Drive'dan yuklab oling yoki Hugging Face Hub'ga push qiling).
2. `.env` ga uchta o'zgaruvchini yozing (yuqoridagi chiqishdan).
3. `make dev` → `/readyz` javobida `"classifier": true` ko'rinishi kerak.
4. `eval/run_eval.py` ni golden set bilan ishlatib, klassifikator
   `zero_touch_rate` ga qancha ta'sir qilganini o'lchang.

**Buni qachon qayta o'rgatish kerak:** yangi hujjat turi qo'shilganda,
yoki eval'da `UNKNOWN_DOC_TYPE` xatosi 5% dan oshganda. Aniqlik yaxshi
bo'lsa — tegmang. Yetarli darajadagi modelni yaxshilashga urinish bu
loyihada vaqtni yo'qotishning eng keng tarqalgan usuli.
